In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv(
    "../data/weather_cleaned.csv",
    parse_dates=["datetime"]
)

In [5]:
df.head()

,T2M,RH2M,PS,datetime
0,7.60,66.63,995.1,2023-01-01 00:00:00
1,7.01,68.41,994.9,2023-01-01 01:00:00
2,6.95,68.69,994.9,2023-01-01 02:00:00
3,6.31,70.77,994.6,2023-01-01 03:00:00
4,6.12,71.02,994.5,2023-01-01 04:00:00


In [6]:
df_anomaly = df.copy()

df_anomaly["is_anomaly"] = 0
df_anomaly["anomaly_type"] = "normal"

In [7]:
df_anomaly.head()

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type
0,7.60,66.63,995.1,2023-01-01 00:00:00,0,normal
1,7.01,68.41,994.9,2023-01-01 01:00:00,0,normal
2,6.95,68.69,994.9,2023-01-01 02:00:00,0,normal
3,6.31,70.77,994.6,2023-01-01 03:00:00,0,normal
4,6.12,71.02,994.5,2023-01-01 04:00:00,0,normal


In [8]:
np.random.seed(42)

spike_idx = np.random.choice(
    df_anomaly.index,
    size=100,
    replace=False
)

df_anomaly.loc[spike_idx, 'T2M'] += np.random.uniform(
    15, 30, size=100
)

df_anomaly.loc[spike_idx, 'is_anomaly'] = 1
df_anomaly.loc[spike_idx, 'anomaly_type'] = 'temperature_spike'

In [12]:
df_anomaly[df_anomaly['anomaly_type'] == 'temperature_spike'].head()

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type
1288,50.952634,23.54,986.7,2023-02-23 16:00:00,1,temperature_spike
1530,43.679098,42.36,991.5,2023-03-05 18:00:00,1,temperature_spike
1788,58.882693,17.34,985.3,2023-03-16 12:00:00,1,temperature_spike
2518,55.853319,32.65,983.3,2023-04-15 22:00:00,1,temperature_spike
2622,46.711033,60.09,979.5,2023-04-20 06:00:00,1,temperature_spike


In [14]:
df.loc[spike_idx[:5], ['datetime', 'T2M']]

,datetime,T2M
12810,2024-06-17 18:00:00,42.81
4408,2023-07-03 16:00:00,38.56
25992,2025-12-19 00:00:00,11.97
21808,2025-06-27 16:00:00,38.60
9579,2024-02-04 03:00:00,12.12


In [15]:
df_anomaly.loc[spike_idx[:5], ['datetime', 'T2M']]

,datetime,T2M
12810,2024-06-17 18:00:00,69.074599
4408,2023-07-03 16:00:00,64.222850
25992,2025-12-19 00:00:00,30.027280
21808,2025-06-27 16:00:00,66.303851
9579,2024-02-04 03:00:00,31.339422


In [16]:
freeze_starts = np.random.choice(
    df_anomaly.index[:-12],
    size=10,
    replace=False
)

for start in freeze_starts:
    value = df_anomaly.loc[start, 'T2M']

    df_anomaly.loc[start:start+11, 'T2M'] = value
    df_anomaly.loc[start:start+11, 'is_anomaly'] = 1
    df_anomaly.loc[start:start+11, 'anomaly_type'] = 'temperature_frozen'

In [17]:
start = freeze_starts[0]

df_anomaly.loc[
    start-3:start+14,
    ['datetime', 'T2M', 'is_anomaly', 'anomaly_type']
]

,datetime,T2M,is_anomaly,anomaly_type
10908,2024-03-30 12:00:00,38.15,0,normal
10909,2024-03-30 13:00:00,38.15,0,normal
10910,2024-03-30 14:00:00,37.62,0,normal
10911,2024-03-30 15:00:00,36.77,1,temperature_frozen
10912,2024-03-30 16:00:00,36.77,1,temperature_frozen
10913,2024-03-30 17:00:00,36.77,1,temperature_frozen
10914,2024-03-30 18:00:00,36.77,1,temperature_frozen
10915,2024-03-30 19:00:00,36.77,1,temperature_frozen
10916,2024-03-30 20:00:00,36.77,1,temperature_frozen
10917,2024-03-30 21:00:00,36.77,1,temperature_frozen


In [18]:
drift_starts = np.random.choice(
    df_anomaly.index[:-24],
    size=10,
    replace=False
)

for start in drift_starts:
    idx = range(start, start + 24)

    # Error gradually grows from 0 to +10°C
    drift = np.linspace(0, 10, 24)

    df_anomaly.loc[idx, 'T2M'] += drift
    df_anomaly.loc[idx, 'is_anomaly'] = 1
    df_anomaly.loc[idx, 'anomaly_type'] = 'temperature_drift'

In [19]:
start = drift_starts[0]

df_anomaly.loc[
    start:start+23,
    ['datetime', 'T2M', 'is_anomaly', 'anomaly_type']
]

,datetime,T2M,is_anomaly,anomaly_type
16495,2024-11-18 07:00:00,14.990000,1,temperature_drift
16496,2024-11-18 08:00:00,17.344783,1,temperature_drift
16497,2024-11-18 09:00:00,20.179565,1,temperature_drift
16498,2024-11-18 10:00:00,24.624348,1,temperature_drift
16499,2024-11-18 11:00:00,27.149130,1,temperature_drift
16500,2024-11-18 12:00:00,28.263913,1,temperature_drift
16501,2024-11-18 13:00:00,28.918696,1,temperature_drift
16502,2024-11-18 14:00:00,28.623478,1,temperature_drift
16503,2024-11-18 15:00:00,27.498261,1,temperature_drift
16504,2024-11-18 16:00:00,25.693043,1,temperature_drift


In [20]:
missing_idx = np.random.choice(
    df_anomaly.index,
    size=100,
    replace=False
)

df_anomaly.loc[missing_idx, 'T2M'] = np.nan
df_anomaly.loc[missing_idx, 'is_anomaly'] = 1
df_anomaly.loc[missing_idx, 'anomaly_type'] = 'communication_error'

In [21]:
df_anomaly[
    df_anomaly['anomaly_type'] == 'communication_error'
].head()

,T2M,RH2M,PS,datetime,is_anomaly,anomaly_type
886,NaN,52.40,986.7,2023-02-06 22:00:00,1,communication_error
1267,NaN,39.68,988.7,2023-02-22 19:00:00,1,communication_error
1359,NaN,19.62,991.5,2023-02-26 15:00:00,1,communication_error
1567,NaN,37.96,991.9,2023-03-07 07:00:00,1,communication_error
1593,NaN,23.14,994.7,2023-03-08 09:00:00,1,communication_error


In [22]:
multi_idx = np.random.choice(
    df_anomaly.dropna().index,
    size=100,
    replace=False
)

source_idx = np.random.choice(
    df.index,
    size=100,
    replace=False
)

# Keep temperature, but replace humidity and pressure
df_anomaly.loc[multi_idx, 'RH2M'] = df.loc[source_idx, 'RH2M'].values
df_anomaly.loc[multi_idx, 'PS'] = df.loc[source_idx, 'PS'].values

df_anomaly.loc[multi_idx, 'is_anomaly'] = 1
df_anomaly.loc[multi_idx, 'anomaly_type'] = 'multivariate_inconsistency'

In [23]:
df_anomaly['anomaly_type'].value_counts()

anomaly_type
normal                        31192
temperature_drift               237
temperature_frozen              119
multivariate_inconsistency      100
communication_error             100
temperature_spike               100
Name: count, dtype: int64

In [24]:
df_anomaly['is_anomaly'].value_counts()

is_anomaly
0    31192
1      656
Name: count, dtype: int64

In [25]:
df_anomaly['is_anomaly'].value_counts(normalize=True) * 100

is_anomaly
0    97.940216
1     2.059784
Name: proportion, dtype: float64

In [26]:
df_anomaly['hour'] = df_anomaly['datetime'].dt.hour
df_anomaly['month'] = df_anomaly['datetime'].dt.month

for col in ['T2M', 'RH2M', 'PS']:
    df_anomaly[f'{col}_diff'] = df_anomaly[col].diff()

    df_anomaly[f'{col}_roll_mean'] = (
        df_anomaly[col].rolling(24).mean()
    )

    df_anomaly[f'{col}_roll_std'] = (
        df_anomaly[col].rolling(24).std()
    )

    df_anomaly[f'{col}_roll_dev'] = (
        df_anomaly[col] - df_anomaly[f'{col}_roll_mean']
    )